# LLM/RAG Initialization

In [ ]:
import pickle

# Load data for the RAG (processed_data.pkl)
with open('../data/processed_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

In [ ]:
type(loaded_data)

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from langchain import PromptTemplate, LLMChain, HuggingFacePipeline
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from rag_source import *
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #This is directory

llm = ChatOpenAI(
    model_name="gpt-5",
    temperature=1, #  o3 model supports only 1
    max_completion_tokens=5000, # max_new_tokens for older models; max_completion_tokens for o3 model
    #request_timeout=300,  # Increase timeout for LLM requests
    # top_p=1.0,
    # frequency_penalty=0.0,
    # presence_penalty=0.0,
    # n=1,
    # stop=None
)

# For o3
# llm = ChatOpenAI(
#     model_name="o3",
#     temperature=1, #  o3 model supports only 1
#     max_completion_tokens=1000, # max_new_tokens for older models; max_completion_tokens for o3 model
#     request_timeout=300,  # Increase timeout for LLM requests
#     # top_p=1.0,
#     # frequency_penalty=0.0,
#     # presence_penalty=0.0,
#     # n=1,
#     # stop=None
# )

# Define a PromptTemplate that accepts prompt_text as an input
prompt_template = PromptTemplate(
    input_variables=["prompt_text"],
    template="{prompt_text}"
)

# Initialize LLMChain with the LLM and prompt template
llm_chain = LLMChain(
    prompt=prompt_template,
    llm=llm
)

# Function to directly analyze the provided text prompt
def analyze_text_prompt(query, processed_data):
    # Pass the prompt text to LLMChain as a dictionary

    prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm_chain.run(inputs)

    return prompt_text, result

## LLM_Retriever Experiment

In [ ]:
from openai import OpenAI
client = OpenAI()

def clean_llm_code(code: str) -> str:
    import re
    code = code.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:python)?\n?", "", code)
        code = re.sub(r"\n?```$", "", code)
    if code.startswith(("'''", '"""')):
        code = code.strip("'''").strip('"""')
    return code.strip()


def llm_code_retrieval(query: str, processed_data: dict, model_name="gpt-4o"):
    """
    Uses GPT to generate Python code that queries the processed_data structure.
    """
    system_prompt = (
        # """
        # You are a data assistant that writes Python code to analyze a structured dictionary named 'loaded_data'. 
        # This dictionary contains eviction trace data (for cache memory). Each value of this dictionary further is a 
        # The keys of the dictionary are trace_ids, in the format '<workload>_evictions_<replacement_policy>'. 
        # dictionary which has three keys: dataframe(the actual dataframe), metadata and description of the workload. 
        # This is the list of possible workloads: astar, lbm, mcf.
        # This is the list of possible replacement policies: belady, lru, mlp, parrot.
        # Within each key, there is a dataframe with eviction data at 'trace_id['data_frame']', metadata at 'trace_id['metadata']' 
        # and policy & workload description at 'trace_id['description']'. Please ensure to use the correct keys, especially dataframe as 'data_frame'.
        # Each dataframe has the following columns - 'program_counter' (PC in hex format), 'memory_address' (Accessed memory address in hex format),
        # 'cache_set_id' (cache set in binary format), 'evict' (Cache Hit/Miss as string), 'miss_type' (Type of cache miss), 'evicted_address' (Address evicted), 
        # 'accessed_address_recency' (Recency in string format), 'accessed_address_reuse_distance' (Number of accesses till the accessed address is accessed again), 
        # 'evicted_address_reuse_distance' (Number of accesses till the evicted address is accessed again),	'function_name' (Function name in source code for PC),	
        # 'function_code' (Code snippet for function at the corresponding PC),	'assembly_code' (Assembly code for the corresponding PC),	
        # 'current_cache_lines' (PC, address pairs in cache set for current cache access),	'recent_access_history' (Past accesses - PC, address pairs - in the trace),
        # 'cache_line_eviction_scores' (Scores for every cache line in the accessed cache set which is used to make eviction decision),	
        # 'current_cache_line_addresses' (Addresses in the set for current access),	'evicted_address_reuse_distance_numeric' (Number of accesses till the evicted address is accessed again),	
        # 'accessed_address_reuse_distance_numeric', 'is_miss' (1 if cache miss, 0 if cache hit),	'accessed_address_recency_numeric' (Number of cache accesses since last access to this memory address).
        # Metadata contains Cache Performance Summary for the complete trace or dataframe: number of - total accesses, total misses, miss rate,
        # capacity misses, conflict misses, total evictions, and wrong evictions where evicted line has lower reuse distance. It also contains 
        # the correlation between accessed address recency and cache misses.
        # Write Python code that returns a list of relevant data points or statistics from this structure 
        # based on the user's natural language question. Make sure the final data structure is named "result". The result should be in the form of
        # a prompt that can be used to answer the user's question by another LLM. 
        # Also provide any other metadata that you find relevant in the result. Please ensure that the final "result" is in string format.
        # ALWAYS wrap the final answer using `result = <string>`. You must format the result using Python's f-string or string concatenation.
        # For example:
        # result = "The miss rate for PC 0x400d81 is 87.3% in the MLP policy."
        # Do not assign DataFrames, dicts, or lists directly to `result`.
        # """
        """
        You are a Python code-writing assistant for analyzing cache memory trace data.
        Your task is to generate Python code that extracts **string-formatted answers** from a dictionary named `loaded_data`.
        ---
        📦 DATA STRUCTURE OVERVIEW:
        - `loaded_data`: a dictionary.
        - Keys: strings like "<workload>_evictions_<replacement_policy>", e.g. "lbm_evictions_lru"
        - Values: dictionaries with the following keys:
            - `data_frame` → a pandas DataFrame (NOT a list)
            - `metadata`   → a descriptive string
            - `description` → a short string describing the workload and policy
        - List of Workloads: astar, lbm, mcf
        - List of Replacement Policies: belady, lru, mlp, parrot
        ---
        🧠 DATAFRAME STRUCTURE (`data_frame`):
        Each `data_frame` is a **pandas DataFrame** with the following columns, described in the format - column name, data structure, and description:
        - `program_counter` — string (hex) — used to identify program instructions e.g. "0x401d9b"
        - `memory_address` — string (hex) — used to identify memory location e.g. "0x35e798a637f"
        - `cache_set_id` — string (binary) — identifies the cache set accessed
        - `evict` — string — eviction decision eg:"Cache Miss" or "Cache Hit"
        - `miss_type` — string — type of miss (e.g., "Capacity Miss", "Conflict Miss")
        - `evicted_address` — string — the memory address that was evicted, if any
        - `accessed_address_recency` — string —
        - `accessed_address_reuse_distance` — numpy.float64 — number of accesses until the accessed address is reused
        - `evicted_address_reuse_distance` — numpy.float64 — number of accesses until the evicted address is reused
        - `function_name` — string — name of the function where the PC is located in source code
        - `function_code` — string — code snippet for the function at the corresponding PC
        - `assembly_code` — string — assembly code for the corresponding PC
        - `current_cache_lines` — list of list of strings — list of tuples (PC, address) pairs in the cache set for current access
        - `recent_access_history` — string — list of tuples (PC, address) pairs in
        - `cache_line_eviction_scores` — string — list of scores for every cache line in the accessed cache set used to make eviction decision
        - `current_cache_line_addresses` — list — addresses in the set for current access
        - `evicted_address_reuse_distance_numeric` — numpy.float64 — number of accesses until the evicted address is reused
        - `accessed_address_reuse_distance_numeric` — numpy.float64 — number of accesses until the accessed address is reused
        - `accessed_address_recency_numeric` — numpy.float64 — number of cache accesses since last access to this memory address
        - `is_miss` — numpy.int64 — 1 = miss, 0 = hit
        ⚠️ Do NOT treat the DataFrame as a list of dictionaries. Use `df[...]` and standard pandas syntax.
        ---
        📊 METADATA (`metadata`):
        - Text summary of cache statistics like total accesses, misses, evictions, miss rate, correlations, etc. for the entire trace in string format.
        - Access via `loaded_data[trace_id]["metadata"]`
        - Metadata is a single string. Extract numbers using string matching or regex like: `re.search(r"(\d+) total misses", metadata)`
        - Example:      
        `Cache Performance Summary: 140704 total accesses, 133542 total misses, 94.91% miss rate,
        100.00% capacity misses, 0.00% conflict misses, 133478 total evictions, 
        87085 (65.24%) wrong evictions where evicted line has lower reuse distance.
        The correlation between accessed address recency and cache misses is 0.18.`
        ---
        🧠 APPLICATION-SPECIFIC INSTRUCTIONS:
        - For questions asking cache hit or cache miss, if there are multiple matching rows, 
        report whether the majority of accesses were hits or misses. For example, if 7 out of 10 accesses were misses, report it as a miss.
        - For questions where you need to use metadata, print all the metadata available in the `metadata` field. If metadata from multiple traces is needed,
        concatenate them into a single string.
        ---
        🎯 TASK INSTRUCTIONS:
        - You must always try to extract relevant information from `loaded_data` based on the user query.
        - Start by checking whether the relevant workload/policy exists.
            - If it does not exist, check for the PC or memory address in all dataframes to find a match. If
            no match is found, follow the steps for "no exact match is found".
        - If a specific PC or memory address is mentioned, search for rows that match. Use:
            df[df['program_counter'] == <pc>]
            or
            df.query('program_counter == "0x401e31" and memory_address == "0x35e798a637f"')
        - If no exact match is found, continue by:
            - Matching just the PC or just the address
            - Using metadata to estimate the answer (e.g., miss rate)
            - Providing cache policy summary from the `metadata` field
        - Never return "No data found" without first checking all these possibilities.
        - If a match is found, include:
            - Eviction result (hit/miss)
            - PC stats
            - Any relevant reuse or recency distances
            - Assembly or function name if available
        - If truly no match exists, return a clear result string like:
            result = "No data found for PC 0x401e31 in workload lbm using policy LRU. Please try a different PC or workload."
        ---
        🧑‍💻 YOUR CODE INSTRUCTIONS:
        - Your output must be **Python code only**.
        - Do not explain your code or output.
        - Do not use markdown, triple quotes, or comments.
        - Your output should assign a final **string** to a variable named `result`.
        - Use f-strings or string concatenation to build the final `result`.
        ---
        ✅ VALID OUTPUT EXAMPLES:
        result = f"The miss rate for PC 0x401e31 in lbm_evictions_LRU is 44.69%."
        result = "According to the metadata, the conflict misses were 14.30% of total accesses."
        ❌ INVALID OUTPUT EXAMPLES:
        result = df ❌
        return df['miss_rate'] ❌
        print(result) ❌
        ---
        🔍 ENHANCED CONTEXT GUIDANCE:
        In addition to directly answering the user query, include the following:
        - Assembly code for the given PC
        - Function name or source-level code for the given PC
        - Metadata summary (e.g., miss rate, conflict/capacity misses) for the given PC, or for the entire trace if no specific PC is given.
        - Description of the workload and replacement policy
        ---
        🎯 FINAL REQUIREMENTS:
        - Use the key "data_frame" exactly — do not misspell it.
        - ALWAYS assign the final answer to a variable named result, and ensure it is a str.
        - You may extract values, format them, and include metadata in the answer string.
        - If no data matches the user query, return a clear message as a string.
        """
        # ---
        # 🧠 DOMAIN KNOWLEDGE: CACHE MEMORY ANALYSIS
        # You are working with cache eviction trace data from CPU simulations. The key goals in this domain include:
        # - Identifying **cache hits and misses**
        # - Measuring **miss rates**, **reuse distances**, and **recency**
        # - Analyzing whether an eviction was **correct** (i.e., evicted line had higher reuse distance than inserted one)
        # - Understanding the behavior of **replacement policies** (e.g., Belady, LRU, MLP, Parrot)
        # Important domain-specific concepts:
        # - **Reuse Distance**: Number of accesses until a memory line is reused again. Lower is better.
        # - **Recency**: Number of accesses since a line was last seen. Lower suggests more temporal locality.
        # - **Bad Evictions**: An eviction is considered poor if the evicted line had a lower reuse distance than the inserted line.
        # - **Belady’s Policy**: Optimal policy based on future access — used as a gold standard for comparison.
        # The data is used to evaluate and compare policies under various workloads like `astar`, `lbm`, and `mcf`.
        # When extracting data:
        # - Focus on values that **support architectural insights**
        # - Prioritize statistics that explain **performance trends or anomalies**
        # - Include context like PC, memory address, and function name when available
    )

    user_prompt = f"User's question: {query}\n Write only the Python code. No explanations. Directly write the code and don't write any comments. Don't mention python either."

    # Step 1: Get code from LLM
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
    )
    
    generated_code = response.choices[0].message.content.strip()

    # Step 2: Prepare a local namespace for safe execution
    local_vars = {"loaded_data": processed_data}

    generated_code = clean_llm_code(response.choices[0].message.content)

    try:
        exec(generated_code, {}, local_vars)
        # You can customize what object the LLM should assign its result to
        result = local_vars.get("result", None)
    except Exception as e:
        result = f"Code execution failed: {str(e)}\n\nGenerated Code:\n{generated_code}"
    
    return result, generated_code


In [ ]:
####### Example usage #######
query = "For the cache access with PC 0x405832 and address 0x210528d2d9d on the astar workload with beladys replacement policy, does the cache hit or miss?"
result, code = llm_code_retrieval(query, processed_data=loaded_data)

print("Generated Code:\n", code)
print("\nRetrieved Result:\n", result)

In [ ]:
def prompt_gen(result,query):
    # Prepare the prompt text for the LLM
    prompt_text = "You are an expert in computer architecture and your job is to answer questions given data from cache traces. "
    prompt_text += "Base your response on the following data and your knowledge of computer architecture.\n\n"
    
    prompt_text += "<CONTEXT>\n"
    prompt_text += "The following data is relevant to the question:\n"
    prompt_text += result + "\n"
    prompt_text += "</CONTEXT>\n"
    
    prompt_text += "Based only on the above context, answer the following question with technical precision.\n" # Only use the information in the context, but feel free to reason about what's given. \n"
    prompt_text += f"\nUser's question: {query}\n"
    prompt_text += "The correct answer is, "
    
    return prompt_text

def analyze_text_prompt_llm(query, processed_data):
    result, code = llm_code_retrieval(query, processed_data)
    
    if not isinstance(result, str):
        try:
            result = str(result)
        except Exception as e:
            result = f"[Unable to convert result to string: {str(e)}]"
    
    prompt_text = prompt_gen(str(result),query)

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm_chain.run(inputs)

    return prompt_text, result

In [ ]:
######### MOCKINGJAY ##########

from openai import OpenAI
client = OpenAI()

def clean_llm_code(code: str) -> str:
    import re
    code = code.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:python)?\n?", "", code)
        code = re.sub(r"\n?```$", "", code)
    if code.startswith(("'''", '"""')):
        code = code.strip("'''").strip('"""')
    return code.strip()

def _load_no_header_csv(path: str):
    import pandas as pd
    df = pd.read_csv(path, header=None)
    if df.shape[1] != 6:
        raise ValueError(f"Expected 6 columns, got {df.shape[1]}")
    df.columns = ["pc", "addr", "set", "etr", "rdp", "hit"]
    return df

def _normalize_single_trace(processed_data):
    import pandas as pd

    if isinstance(processed_data, str):
        df = _load_no_header_csv(processed_data)
        return {"data_frame": df, "metadata": "", "description": ""}

    if isinstance(processed_data, pd.DataFrame):
        df = processed_data.copy()
        if df.columns.tolist() == list(range(df.shape[1])) and df.shape[1] == 6:
            df.columns = ["pc", "addr", "set", "etr", "rdp", "hit"]
        return {"data_frame": df, "metadata": "", "description": ""}

    if isinstance(processed_data, dict):
        df = None
        if "data_frame" in processed_data:
            df = processed_data["data_frame"]
        elif "df" in processed_data:
            df = processed_data["df"]
        else:
            raise ValueError("processed_data is a dict, but no 'data_frame' or 'df' key was found.")

        if isinstance(df, str):
            df = _load_no_header_csv(df)

        if not isinstance(df, pd.DataFrame):
            raise TypeError("The provided 'data_frame'/'df' must be a pandas DataFrame or a CSV filepath.")

        df = df.copy()
        if df.columns.tolist() == list(range(df.shape[1])) and df.shape[1] == 6:
            df.columns = ["pc", "addr", "set", "etr", "rdp", "hit"]

        return {
            "data_frame": df,
            "metadata": processed_data.get("metadata", ""),
            "description": processed_data.get("description", "")
        }

    raise TypeError("processed_data must be a DataFrame, dict, or CSV filepath (str).")

def llm_code_retrieval(query: str, processed_data, model_name="gpt-4o"):
    import pandas as pd

    loaded_data = _normalize_single_trace(processed_data)
    df = loaded_data["data_frame"]

    if not isinstance(df, pd.DataFrame):
        raise TypeError("loaded_data['data_frame'] must be a pandas DataFrame.")
    if set(df.columns) != {"pc", "addr", "set", "etr", "rdp", "hit"}:
        raise ValueError(f"Unexpected columns: {df.columns.tolist()} (expected pc, addr, set, etr, rdp, hit)")

    system_prompt = """
You are a Python code-writing assistant for analyzing a SINGLE cache trace.

A dictionary named `loaded_data` exists in scope:
- loaded_data["data_frame"] is a pandas DataFrame with EXACT columns:
  - pc, addr, set, etr, rdp, hit

CSV-BASED DATA TYPES (IMPORTANT):
- pc: string, lowercase hex WITHOUT "0x" prefix (example: "41807d")
- addr: string, lowercase hex WITHOUT "0x" prefix (example: "a48b819b7e00")
- set: integer (example: 1528)
- etr: integer (example: 4)
- rdp: integer (example: 127)
- hit: integer 0 or 1 (1 = cache hit, 0 = cache miss)

TASK:
- Generate Python code that answers the user's question using pandas.
- The final output MUST be a string assigned to variable `result`.
- Do not print. Do not return.

NORMALIZATION RULES FOR MATCHING USER QUERIES:
- Users may provide pc/addr with or without "0x", and in upper/lowercase.
- Your code should normalize inputs before filtering:
  - strip whitespace
  - lowercase
  - remove leading "0x" if present
Example:
  pc_query = pc_query.strip().lower()
  if pc_query.startswith("0x"): pc_query = pc_query[2:]

FILTERING GUIDANCE:
- If asked about a specific pc, filter like:
  sub = df[df["pc"] == pc_query]
- If asked about a specific addr:
  sub = df[df["addr"] == addr_query]
- If asked about both:
  sub = df[(df["pc"] == pc_query) & (df["addr"] == addr_query)]
- If no exact match exists, fall back in this order:
  1) match only pc
  2) match only addr
  3) global stats

HIT/MISS METRICS:
- hit_rate = mean(df["hit"] == 1)
- miss_rate = mean(df["hit"] == 0)
- For subsets, compute the same way.

USEFUL SUMMARIES TO INCLUDE WHEN RELEVANT:
- counts: total rows, #hits, #misses
- hit/miss rate
- top cache sets for a given pc/addr: sub["set"].value_counts().head(k)
- mean/median etr and rdp for a given subset
- if asked about ETR/RDP distributions: describe() or quantiles

OUTPUT REQUIREMENTS:
- Output Python code only.
- No markdown, no comments, no triple quotes.
- The last line must set: result = <string>
"""

    user_prompt = (
        f"User's question: {query}\n"
        "Write only Python code. No explanations. No comments. "
        "Make sure the final answer is assigned to a string variable named result."
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.8,
    )

    generated_code = clean_llm_code(response.choices[0].message.content)

    local_vars = {"loaded_data": loaded_data}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", None)
        if not isinstance(result, str):
            result = f"Code ran but `result` was not a string. Got: {type(result)}"
    except Exception as e:
        result = f"Code execution failed: {str(e)}\n\nGenerated Code:\n{generated_code}"

    return result, generated_code

In [ ]:
import pandas as pd
df = pd.read_csv("/home/kmhapse/Imitation-Learning/cache_replacement/ChampSim_Fixed/ChampSim/milc_360_200_1000_mockingjay.csv", header=None)

In [ ]:
result, code = llm_code_retrieval("For PC 413a5b, what fraction are misses and what is mean ETR?", df)

print("Generated Code:\n", code)
print("\nRetrieved Result:\n", result)

In [ ]:
######### MOCKINGJAY CHATBOT ###########


import warnings
warnings.filterwarnings("ignore")

from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOpenAI


def prompt_gen(context_result: str, query: str) -> str:
    prompt_text = (
        "You are an expert in computer architecture and your job is to answer questions "
        "given data from cache traces. Base your response ONLY on the context.\n\n"
        "<CONTEXT>\n"
        f"{context_result}\n"
        "</CONTEXT>\n\n"
        "Based only on the above context, answer the following question with technical precision.\n"
        f"User's question: {query}\n"
        "The correct answer is: "
    )
    return prompt_text


def build_llm_chain_with_memory(model_name="gpt-5", temperature=1, max_tokens=5000):
    llm = ChatOpenAI(
        model_name=model_name,
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )

    memory = ConversationBufferMemory(memory_key="history", return_messages=True)

    # We pass the whole prompt as a variable, and let memory add history.
    prompt = PromptTemplate(
        input_variables=["history", "prompt_text"],
        template=(
            "Conversation so far:\n{history}\n\n"
            "{prompt_text}"
        ),
    )

    llm_chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=False)
    return llm_chain


def chatbot_loop_mockingjay(processed_data, model_name="gpt-5"):
    llm_chain = build_llm_chain_with_memory(model_name=model_name)

    print("🔁 Mockingjay Trace Chatbot (LLM Retriever + Memory). Type 'exit' to stop.")
    while True:
        query = input("\nUser: ").strip()
        if query.lower() in {"exit", "quit", "bye"}:
            print("👋 Exiting the chatbot. Goodbye!")
            break

        # 1) Retrieval (your code-gen retriever)
        context_result, generated_code = llm_code_retrieval(query, processed_data)

        if not isinstance(context_result, str):
            try:
                context_result = str(context_result)
            except Exception as e:
                context_result = f"[Unable to convert context_result to string: {e}]"

        # Optional: debug
        # print("\n[DEBUG] Generated retriever code:\n", generated_code)

        # 2) Build final prompt for answer model
        prompt_text = prompt_gen(context_result, query)

        # 3) Answer with memory
        try:
            response = llm_chain.run({"prompt_text": prompt_text})
        except Exception as e:
            response = f"Generation failed: {e}"

        print("\nUser:", query)
        print("\nAssistant:", response)


# Example usage:
# processed_data can be: df, dict, or csv path depending on your llm_code_retrieval normalizer
# chatbot_loop_mockingjay(df)

In [ ]:
chatbot_loop_mockingjay(df)

In [ ]:
for m in memory.chat_memory.messages:
    print(f"{m.type.upper()}: {m.content}")

In [ ]:
############# GEM5 CHATBOT #############

from openai import OpenAI
client = OpenAI()

def clean_llm_code(code: str) -> str:
    import re
    code = code.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:python)?\n?", "", code)
        code = re.sub(r"\n?```$", "", code)
    if code.startswith(("'''", '"""')):
        code = code.strip("'''").strip('"""')
    return code.strip()

def _load_gem5_csv(path: str):
    import pandas as pd

    df = pd.read_csv(path)  # header is present
    # Normalize column names
    df.columns = [c.strip().lower() for c in df.columns]

    required = {"pc","instruction","cache","type","addrlo","addrhi","hitmiss","set","way"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"gem5 CSV missing columns. Found: {df.columns.tolist()}")

    # Rename to stable internal names
    df = df.rename(columns={
        "instruction": "inst",
        "type": "req_type",
        "addrlo": "addr_lo",
        "addrhi": "addr_hi",
        "hitmiss": "hitmiss",
    })

    # Force string columns (gem5 can mix hex/NA)
    for c in ["pc","inst","cache","req_type","addr_lo","addr_hi","hitmiss","set","way"]:
        df[c] = df[c].astype(str)

    # Normalize obvious text
    df["pc"] = df["pc"].str.strip().str.lower()
    df["addr_lo"] = df["addr_lo"].str.strip().str.lower()
    df["addr_hi"] = df["addr_hi"].str.strip().str.lower()
    df["hitmiss"] = df["hitmiss"].str.strip().str.lower()
    df["cache"] = df["cache"].str.strip()
    df["req_type"] = df["req_type"].str.strip()

    # Create numeric hit column (1=hit, 0=miss, NaN otherwise)
    def _hm_to_hit(x: str):
        x = str(x).strip().lower()
        if x == "hit":
            return 1
        if x == "miss":
            return 0
        return None

    df["hit"] = df["hitmiss"].map(_hm_to_hit)

    return df

def _normalize_single_trace(processed_data):
    import pandas as pd

    if isinstance(processed_data, str):
        df = _load_gem5_csv(processed_data)
        return {"data_frame": df, "metadata": "", "description": ""}

    if isinstance(processed_data, pd.DataFrame):
        df = processed_data.copy()
        # If user already loaded gem5 CSV, normalize column names similarly
        df.columns = [c.strip().lower() for c in df.columns]
        # If it looks like gem5 format, rename and add hit
        if {"pc","instruction","cache","type","addrlo","addrhi","hitmiss","set","way"}.issubset(set(df.columns)):
            df = df.rename(columns={
                "instruction": "inst",
                "type": "req_type",
                "addrlo": "addr_lo",
                "addrhi": "addr_hi",
            })
            for c in ["pc","inst","cache","req_type","addr_lo","addr_hi","hitmiss","set","way"]:
                df[c] = df[c].astype(str)
            df["hitmiss"] = df["hitmiss"].astype(str).str.strip().str.lower()
            df["pc"] = df["pc"].astype(str).str.strip().str.lower()
            df["addr_lo"] = df["addr_lo"].astype(str).str.strip().str.lower()
            df["addr_hi"] = df["addr_hi"].astype(str).str.strip().str.lower()

            df["hit"] = df["hitmiss"].map(lambda x: 1 if x=="hit" else (0 if x=="miss" else None))

        return {"data_frame": df, "metadata": "", "description": ""}

    if isinstance(processed_data, dict):
        df = processed_data.get("data_frame", processed_data.get("df", None))
        if df is None:
            raise ValueError("processed_data dict must contain 'data_frame' or 'df'.")

        if isinstance(df, str):
            df = _load_gem5_csv(df)

        if not isinstance(df, pd.DataFrame):
            raise TypeError("The provided 'data_frame'/'df' must be a pandas DataFrame or a CSV filepath.")

        # normalize as above
        return _normalize_single_trace(df) | {
            "metadata": processed_data.get("metadata", ""),
            "description": processed_data.get("description", "")
        }

    raise TypeError("processed_data must be a DataFrame, dict, or CSV filepath (str).")

In [ ]:
def llm_code_retrieval(query: str, processed_data, model_name="gpt-4o"):
    import pandas as pd

    loaded_data = _normalize_single_trace(processed_data)
    df = loaded_data["data_frame"]

    expected = {"pc","inst","cache","req_type","addr_lo","addr_hi","hitmiss","set","way","hit"}
    if not isinstance(df, pd.DataFrame):
        raise TypeError("loaded_data['data_frame'] must be a pandas DataFrame.")
    if set(df.columns) != expected:
        raise ValueError(f"Unexpected columns: {df.columns.tolist()} (expected {sorted(expected)})")

    system_prompt = """
You are a Python code-writing assistant for analyzing a SINGLE gem5 cache trace.

A dictionary named `loaded_data` exists in scope:
- loaded_data["data_frame"] is a pandas DataFrame with EXACT columns:
  - pc, inst, cache, req_type, addr_lo, addr_hi, hitmiss, set, way, hit

CSV-BASED DATA TYPES (IMPORTANT):
- pc: string, hex with optional "0x" prefix (example: "0x4038a8"), already lowercased
- inst: string instruction mnemonic / decoded label (example: "MOV_R_M")
- cache: string cache level (example: "L1", "L2", "LLC" if present)
- req_type: string request type (example: "ReadReq", "ReadSharedReq", "ReadExReq")
- addr_lo: string hex (often without 0x) representing low address / block range start (example: "bde50")
- addr_hi: string hex representing high address / block range end (example: "bde57")
- hitmiss: string "hit" or "miss"
- set: string, either hex like "0x79" or "NA"
- way: string, either hex like "0x0" or "NA"
- hit: numeric 1 (hit) or 0 (miss) or None

TASK:
- Generate Python code that answers the user's question using pandas.
- The final output MUST be a string assigned to variable `result`.
- Do not print. Do not return.

NORMALIZATION RULES FOR MATCHING USER QUERIES:
- Users may provide pc with/without "0x" and in upper/lowercase.
- Normalize any user-provided hex strings:
  - strip whitespace
  - lowercase
  - if it starts with "0x", keep it consistent OR strip it consistently before comparison
- Since df["pc"] includes "0x" in the dataset, prefer normalizing to include "0x".
  Example:
    pcq = pcq.strip().lower()
    if not pcq.startswith("0x"): pcq = "0x" + pcq

FILTERING GUIDANCE:
- Filter by pc:
    sub = df[df["pc"] == pcq]
- Filter by cache level:
    sub = sub[sub["cache"] == "L1"]  (or "L2")
- Filter by hit/miss:
    sub = sub[sub["hit"] == 1]  or  sub[sub["hit"] == 0]
- Address range queries:
  - match addr_lo/addr_hi as strings after lowercasing and stripping optional "0x"
  - or treat them as integers using int(x, 16) when asked for range comparisons

HIT/MISS METRICS:
- hit_rate = mean(df["hit"] == 1) ignoring None
- miss_rate = mean(df["hit"] == 0) ignoring None

USEFUL SUMMARIES TO INCLUDE WHEN RELEVANT:
- counts: total rows, #hits, #misses (use df["hit"].value_counts(dropna=True))
- hit/miss rate overall or for subsets
- top PCs: sub["pc"].value_counts().head(k)
- top sets/ways (exclude "NA"): sub[sub["set"]!="NA"]["set"].value_counts().head(k)
- breakdown by cache level: df.groupby("cache")["hit"].mean()

OUTPUT REQUIREMENTS:
- Output Python code only.
- No markdown, no comments, no triple quotes.
- The last line must set: result = <string>
"""

    user_prompt = (
        f"User's question: {query}\n"
        "Write only Python code. No explanations. No comments. "
        "Make sure the final answer is assigned to a string variable named result."
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.8,
    )

    generated_code = clean_llm_code(response.choices[0].message.content)

    local_vars = {"loaded_data": loaded_data}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", None)
        if not isinstance(result, str):
            result = f"Code ran but `result` was not a string. Got: {type(result)}"
    except Exception as e:
        result = f"Code execution failed: {str(e)}\n\nGenerated Code:\n{generated_code}"

    return result, generated_code

In [ ]:
import pandas as pd
df = pd.read_csv("../data/exp2.csv")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOpenAI


def prompt_gen(context_result: str, query: str) -> str:
    prompt_text = (
        "You are an expert in computer architecture and your job is to answer questions "
        "given data from cache traces. Base your response ONLY on the context.\n\n"
        "<CONTEXT>\n"
        f"{context_result}\n"
        "</CONTEXT>\n\n"
        "Based only on the above context, answer the following question with technical precision.\n"
        f"User's question: {query}\n"
        "The correct answer is: "
    )
    return prompt_text


def build_llm_chain_with_memory(model_name="gpt-5", temperature=1, max_tokens=5000):
    llm = ChatOpenAI(
        model_name=model_name,
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )

    memory = ConversationBufferMemory(memory_key="history", return_messages=True)

    # We pass the whole prompt as a variable, and let memory add history.
    prompt = PromptTemplate(
        input_variables=["history", "prompt_text"],
        template=(
            "Conversation so far:\n{history}\n\n"
            "{prompt_text}"
        ),
    )

    llm_chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=False)
    return llm_chain


def chatbot_loop_gem5(processed_data, model_name="gpt-5"):
    llm_chain = build_llm_chain_with_memory(model_name=model_name)

    print("🔁 Gem5 Trace Chatbot (LLM Retriever + Memory). Type 'exit' to stop.")
    while True:
        query = input("\nUser: ").strip()
        if query.lower() in {"exit", "quit", "bye"}:
            print("👋 Exiting the chatbot. Goodbye!")
            break

        # 1) Retrieval (your code-gen retriever)
        context_result, generated_code = llm_code_retrieval(query, processed_data)

        if not isinstance(context_result, str):
            try:
                context_result = str(context_result)
            except Exception as e:
                context_result = f"[Unable to convert context_result to string: {e}]"

        # Optional: debug
        # print("\n[DEBUG] Generated retriever code:\n", generated_code)

        # 2) Build final prompt for answer model
        prompt_text = prompt_gen(context_result, query)

        # 3) Answer with memory
        try:
            response = llm_chain.run({"prompt_text": prompt_text})
        except Exception as e:
            response = f"Generation failed: {e}"

        print("\nUser:", query)
        print("\nAssistant:", response)

In [ ]:
chatbot_loop_gem5(df)  # you can rename this function to gem5 if you want

In [ ]:
########## GEM5 CHATBOT FOR EXP2 ###########

from openai import OpenAI
client = OpenAI()

def clean_llm_code(code: str) -> str:
    import re
    code = code.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:python)?\n?", "", code)
        code = re.sub(r"\n?```$", "", code)
    if code.startswith(("'''", '"""')):
        code = code.strip("'''").strip('"""')
    return code.strip()


def _normalize_hex_str(s: str, want_0x: bool):
    # Normalizes hex-like strings; leaves non-hex tokens (e.g., "nan", "NA") alone.
    s = str(s).strip().lower()
    if s in {"nan", "none", "na", ""}:
        return "NA"
    if s.startswith("0x"):
        core = s[2:]
    else:
        core = s
    # If it's truly hex digits, normalize prefix as requested
    is_hex = all(c in "0123456789abcdef" for c in core) and core != ""
    if not is_hex:
        return s
    return ("0x" + core) if want_0x else core


def _load_gem5_csv(path: str):
    import pandas as pd

    df = pd.read_csv(path)  # header exists in exp2.csv
    # Strip column names exactly (keep original names)
    df.columns = [c.strip() for c in df.columns]

    expected_cols = ["PC", "Instruction", "Cache", "Type", "AddrLo", "AddrHi", "HitMiss", "Set", "Way"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"exp2.csv missing columns: {missing}. Found: {df.columns.tolist()}")

    df = df[expected_cols].copy()

    # Force everything to string first to avoid pandas NaN surprises in comparisons
    for c in expected_cols:
        df[c] = df[c].astype(str)

    # Normalize fields
    df["PC"] = df["PC"].map(lambda x: _normalize_hex_str(x, want_0x=True))          # keep 0x
    df["AddrLo"] = df["AddrLo"].map(lambda x: _normalize_hex_str(x, want_0x=False))  # no 0x
    df["AddrHi"] = df["AddrHi"].map(lambda x: _normalize_hex_str(x, want_0x=False))  # no 0x

    df["Set"] = df["Set"].map(lambda x: _normalize_hex_str(x, want_0x=True))         # keep 0x, or NA
    df["Way"] = df["Way"].map(lambda x: _normalize_hex_str(x, want_0x=True))         # keep 0x, or NA

    df["HitMiss"] = df["HitMiss"].str.strip().str.lower()
    # Standardize hit/miss tokens
    df.loc[~df["HitMiss"].isin(["hit", "miss"]), "HitMiss"] = "NA"

    df["Instruction"] = df["Instruction"].astype(str).str.strip()
    df["Cache"] = df["Cache"].astype(str).str.strip()
    df["Type"] = df["Type"].astype(str).str.strip()

    # Add numeric hit flag (1 hit, 0 miss, NA otherwise)
    df["hit"] = df["HitMiss"].map(lambda x: 1 if x == "hit" else (0 if x == "miss" else None))

    return df


def _normalize_single_trace(processed_data):
    import pandas as pd

    # Case 1: filepath string
    if isinstance(processed_data, str):
        df = _load_gem5_csv(processed_data)
        return {"data_frame": df, "metadata": "", "description": ""}

    # Case 2: DataFrame
    if isinstance(processed_data, pd.DataFrame):
        df = processed_data.copy()
        df.columns = [c.strip() for c in df.columns]

        expected_cols = ["PC", "Instruction", "Cache", "Type", "AddrLo", "AddrHi", "HitMiss", "Set", "Way"]
        if not all(c in df.columns for c in expected_cols):
            raise ValueError(
                f"DataFrame does not look like exp2.csv. "
                f"Expected columns {expected_cols}, got {df.columns.tolist()}"
            )

        # Apply the same normalization as file-load path
        for c in expected_cols:
            df[c] = df[c].astype(str)

        df["PC"] = df["PC"].map(lambda x: _normalize_hex_str(x, want_0x=True))
        df["AddrLo"] = df["AddrLo"].map(lambda x: _normalize_hex_str(x, want_0x=False))
        df["AddrHi"] = df["AddrHi"].map(lambda x: _normalize_hex_str(x, want_0x=False))
        df["Set"] = df["Set"].map(lambda x: _normalize_hex_str(x, want_0x=True))
        df["Way"] = df["Way"].map(lambda x: _normalize_hex_str(x, want_0x=True))

        df["HitMiss"] = df["HitMiss"].str.strip().str.lower()
        df.loc[~df["HitMiss"].isin(["hit", "miss"]), "HitMiss"] = "NA"

        df["Instruction"] = df["Instruction"].astype(str).str.strip()
        df["Cache"] = df["Cache"].astype(str).str.strip()
        df["Type"] = df["Type"].astype(str).str.strip()

        df["hit"] = df["HitMiss"].map(lambda x: 1 if x == "hit" else (0 if x == "miss" else None))

        return {"data_frame": df, "metadata": "", "description": ""}

    # Case 3: dict wrapper
    if isinstance(processed_data, dict):
        df = processed_data.get("data_frame", processed_data.get("df", None))
        if df is None:
            raise ValueError("processed_data dict must contain 'data_frame' or 'df'.")

        if isinstance(df, str):
            base = _normalize_single_trace(df)
        else:
            base = _normalize_single_trace(df)

        base["metadata"] = processed_data.get("metadata", "")
        base["description"] = processed_data.get("description", "")
        return base

    raise TypeError("processed_data must be a DataFrame, dict, or CSV filepath (str).")


def llm_code_retrieval(query: str, processed_data, model_name="gpt-4o"):
    import pandas as pd

    loaded_data = _normalize_single_trace(processed_data)
    df = loaded_data["data_frame"]

    expected = {"PC","Instruction","Cache","Type","AddrLo","AddrHi","HitMiss","Set","Way","hit"}
    if not isinstance(df, pd.DataFrame):
        raise TypeError("loaded_data['data_frame'] must be a pandas DataFrame.")
    if set(df.columns) != expected:
        raise ValueError(f"Unexpected columns: {df.columns.tolist()} (expected {sorted(expected)})")

    system_prompt = """
You are a Python code-writing assistant for analyzing a SINGLE gem5 cache trace.

A dictionary named `loaded_data` exists in scope:
- loaded_data["data_frame"] is a pandas DataFrame with EXACT columns:
  - PC, Instruction, Cache, Type, AddrLo, AddrHi, HitMiss, Set, Way, hit

DATA TYPES (IMPORTANT):
- PC: string, normalized to lowercase, ALWAYS with "0x" prefix (example: "0x4038a8")
- AddrLo / AddrHi: string, normalized to lowercase hex WITHOUT "0x" (example: "9e38", "9e3f") or "NA"
- Set / Way: string, normalized to lowercase with "0x" prefix (example: "0x78", "0x0") or "NA"
- HitMiss: string, "hit" or "miss" or "NA"
- hit: numeric 1 if hit, 0 if miss, None if NA
- Cache: string (e.g., "L1", "L2")
- Type: string request type (e.g., "ReadReq")
- Instruction: string mnemonic/decoded label (e.g., "MOV_R_M")

TASK:
- Generate Python code that answers the user's question using pandas.
- The final output MUST be a string assigned to variable `result`.
- Do not print. Do not return.

MATCHING USER INPUTS:
- If the user provides a PC like "4038a8", normalize it to "0x4038a8" (lowercase).
- If the user provides Set/Way without 0x, normalize similarly.
- AddrLo/AddrHi are stored WITHOUT 0x; strip 0x if the user includes it.

FILTERING GUIDANCE:
- By PC:
    sub = df[df["PC"] == pcq]
- By Cache level:
    sub = sub[sub["Cache"] == "L1"]  (or "L2")
- By hit/miss:
    sub = sub[sub["HitMiss"] == "hit"]  or  sub[sub["HitMiss"] == "miss"]
- By Set/Way when present:
    sub2 = sub[(sub["Set"] == setq) & (sub["Way"] == wayq)]
  If Set/Way are "NA", do NOT require them; fall back to other filters.

HIT/MISS METRICS:
- Overall hit rate (ignoring NA):
    valid = df[df["HitMiss"].isin(["hit","miss"])]
    hit_rate = (valid["HitMiss"] == "hit").mean()
- Similar for subsets.

USEFUL SUMMARIES:
- total accesses, hits, misses, hit/miss rate
- hit rate by Cache: groupby("Cache")
- top miss PCs: df[df["HitMiss"]=="miss"]["PC"].value_counts().head(k)
- for a PC: breakdown by Cache and Type
- sets/ways only where Set != "NA"

OUTPUT REQUIREMENTS:
- Python code only.
- No markdown, no comments, no triple quotes.
- The last line must set: result = <string>
"""

    user_prompt = (
        f"User's question: {query}\n"
        "Write only Python code. No explanations. No comments. "
        "Make sure the final answer is assigned to a string variable named result."
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.8,
    )

    generated_code = clean_llm_code(response.choices[0].message.content)

    local_vars = {"loaded_data": loaded_data}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", None)
        if not isinstance(result, str):
            result = f"Code ran but `result` was not a string. Got: {type(result)}"
    except Exception as e:
        result = f"Code execution failed: {str(e)}\n\nGenerated Code:\n{generated_code}"

    return result, generated_code

In [ ]:
df = _load_gem5_csv("../data/exp2.csv")

In [ ]:
chatbot_loop_gem5(df) 

In [ ]:
from docx import Document
import pandas as pd
import openai
import os

# Set your OpenAI key (or assume it's set via env var)
openai.api_key = os.getenv("OPENAI_API_KEY")

# === STEP 1: Load Q&A from DOCX ===
def load_questions_from_docx(file_path):
    doc = Document(file_path)
    qna_pairs = []

    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for i in range(0, len(paragraphs) - 1, 2):  # assuming Q on even lines, A on odd
        question = paragraphs[i]
        expected = paragraphs[i + 1]
        qna_pairs.append((question, expected))
    
    return qna_pairs

# === STEP 2: Query GPT ===
def ask_gpt(question): #, model="gpt-4"):
    try:
        # response = openai.ChatCompletion.create(
        #     model=model,
        #     messages=[
        #         {"role": "system", "content": "You are a helpful and precise assistant."},
        #         {"role": "user", "content": question}
        #     ]
        # )
        # return response['choices'][0]['message']['content'].strip()
        prompt, response = analyze_text_prompt_llm(question, loaded_data)
        # Remove the prompt_text from the beginning of result
        if response.startswith(prompt):
            response = response[len(prompt):]
        return prompt, response
    except Exception as e:
        return question, f"[ERROR] {str(e)}"

# === STEP 3: Run All ===
def process_questions(docx_path, output_csv="../data/llm_rag_"+llm_chain.llm.model_name+"_eval_output.csv"):
    qna_pairs = load_questions_from_docx(docx_path)

    results = []
    for idx, (question, expected) in enumerate(qna_pairs, 1):
        print(f"Processing {idx}/{len(qna_pairs)}...")
        prompt, model_answer = ask_gpt(question)
        results.append({
            "Question": question,
            "Expected Answer": expected,
            "Model Answer": model_answer,
            "Prompt": prompt,
            "Evaluation": ""  # Leave blank for manual input
        })

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Done. Saved results to {output_csv}")

# === USAGE ===
# Replace with your actual DOCX file
process_questions("../data/10_questions.docx")
# process_questions("../data/Benchmark Questions_GPT.docx")

In [ ]:
# from langchain_community.chat_models import ChatOpenAI
# from langchain import PromptTemplate, LLMChain, HuggingFacePipeline
# from sentence_transformers import SentenceTransformer, util
# from transformers import pipeline
# import torch
# import re
# from rag_source import *
# import pandas as pd
# import numpy as np
# import warnings
# warnings.filterwarnings('ignore')

# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #This is directory

# llm = ChatOpenAI(
#     model_name="gpt-4o-mini",
#     temperature=1, #  o3 model supports only 1
#     max_completion_tokens=1000, # max_new_tokens for older models; max_completion_tokens for o3 model
#     # top_p=1.0,
#     # frequency_penalty=0.0,
#     # presence_penalty=0.0,
#     # n=1,
#     # stop=None
# )

# # Define a PromptTemplate that accepts prompt_text as an input
# prompt_template = PromptTemplate(
#     input_variables=["prompt_text"],
#     template="{prompt_text}"
# )

# # Initialize LLMChain with the LLM and prompt template
# llm_chain = LLMChain(
#     prompt=prompt_template,
#     llm=llm
# )

# # Function to directly analyze the provided text prompt
# def analyze_text_prompt(query, processed_data):
#     # Pass the prompt text to LLMChain as a dictionary

#     prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

#     inputs = {
#         "prompt_text": prompt_text
#     }
    
#     # Run the LLM chain and get the result
#     result = llm_chain.run(inputs)

#     return prompt_text, result


In [ ]:
query = "For astar + Parrot, which memory address is evicted most frequently?"
#For the cache access with PC 0x403a85 and address 0x35e798a637f on the bzip workload with PARROT replacement policy, does the cache hit or miss?
prompt, response = analyze_text_prompt(query, loaded_data)

# Remove the prompt_text from the beginning of result
if response.startswith(prompt):
    response = response[len(prompt):]

print("\nPrompt: ", prompt)

## 🔍 Zero-Shot Prompt Setup

This section builds a simple zero-shot `PromptTemplate` that just takes the raw prompt text, without any examples.

We then create a `LLMChain` that connects this template to our LLM. This is our baseline zero-shot approach.


## 🏗 Running Zero-Shot Inference

We use our `analyze_text_prompt` function to:
- Process the query and context.
- Pass it through the `LLMChain`.
- Get a direct answer without any examples (zero-shot).

We then print out the prompt and the model's generated response.


In [ ]:
from docx import Document
import pandas as pd
import openai
import os

# Set your OpenAI key (or assume it's set via env var)
openai.api_key = os.getenv("OPENAI_API_KEY")

# === STEP 1: Load Q&A from DOCX ===
def load_questions_from_docx(file_path):
    doc = Document(file_path)
    qna_pairs = []

    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for i in range(0, len(paragraphs) - 1, 2):  # assuming Q on even lines, A on odd
        question = paragraphs[i]
        expected = paragraphs[i + 1]
        qna_pairs.append((question, expected))
    
    return qna_pairs

# === STEP 2: Query GPT ===
def ask_gpt(question): #, model="gpt-4"):
    try:
        # response = openai.ChatCompletion.create(
        #     model=model,
        #     messages=[
        #         {"role": "system", "content": "You are a helpful and precise assistant."},
        #         {"role": "user", "content": question}
        #     ]
        # )
        # return response['choices'][0]['message']['content'].strip()
        prompt, response = analyze_text_prompt(question, loaded_data)
        # Remove the prompt_text from the beginning of result
        if response.startswith(prompt):
            response = response[len(prompt):]
        return prompt, response
    except Exception as e:
        return question, f"[ERROR] {str(e)}"

# === STEP 3: Run All ===
def process_questions(docx_path, output_csv="../data/"+llm_chain.llm.model_name+"_eval_output.csv"):
    qna_pairs = load_questions_from_docx(docx_path)

    results = []
    for idx, (question, expected) in enumerate(qna_pairs, 1):
        print(f"Processing {idx}/{len(qna_pairs)}...")
        prompt, model_answer = ask_gpt(question)
        results.append({
            "Question": question,
            "Expected Answer": expected,
            "Model Answer": model_answer,
            "Prompt": prompt,
            "Evaluation": ""  # Leave blank for manual input
        })

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Done. Saved results to {output_csv}")

# === USAGE ===
# Replace with your actual DOCX file
process_questions("../data/Benchmark Questions_GPT.docx")

In [ ]:
query = "Which replacement policy yields the lowest overall miss rate on the lbm workload?"
#For the cache access with PC 0x403a85 and address 0x35e798a637f on the bzip workload with PARROT replacement policy, does the cache hit or miss?
prompt, response = analyze_text_prompt(query, loaded_data)

# Remove the prompt_text from the beginning of result
if response.startswith(prompt):
    response = response[len(prompt):]

print("\nPrompt: ", prompt)
print("\nResponse: ", response)

# Evaluating LLMs

In [ ]:
import pandas as pd
import re
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from moverscore import get_idf_dict, word_mover_score
from tqdm import tqdm
import openai
import nltk
import os
import time

# Ensure nltk data is available
# nltk.download("punkt")

# Load CSV
df = pd.read_csv("../data/gpt-4o-mini_eval_output.csv")

# === Step 1: Clean "Ans:" from expected answers ===
df["Expected Answer"] = (
    df["Expected Answer"]
    .astype(str)
    .str.replace(r"(?i)^ans\s*:\s*", "", regex=True)
    .str.strip('"')
    .str.strip()
)

# Also clean model answers just in case
df["Model Answer"] = df["Model Answer"].astype(str).str.strip('"').str.strip()

# references = df["Expected Answer"].tolist()
# candidates = df["Model Answer"].tolist()

# Filter for safe BERTScore inputs
references = []
candidates = []
valid_indices = []

for i, (ref, cand) in enumerate(zip(df["Expected Answer"], df["Model Answer"])):
    if isinstance(ref, str) and isinstance(cand, str) and ref.strip() and cand.strip():
        references.append(ref)
        candidates.append(cand)
        valid_indices.append(i)

# === Step 2: BERT Score ===
# P, R, F1 = bert_score(candidates, references, lang="en", verbose=True)
# df["BERT Score"] = F1

# BERTScore (wrapped in try-except and on CPU)
try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = bert_score(candidates, references, lang="en", verbose=True, device='cpu')

    # Apply scores only to valid rows
    df["BERT Score"] = np.nan
    for idx, f1 in zip(valid_indices, F1):
        df.at[idx, "BERT Score"] = f1.item() if hasattr(f1, "item") else f1

except Exception as e:
    print(f"Error computing BERTScore: {e}")


# === Step 3: BLEU Score ===
# smoothie = SmoothingFunction().method4
# df["BLEU Score"] = [
#     sentence_bleu([ref.split()], cand.split(), smoothing_function=smoothie)
#     for ref, cand in zip(references, candidates)
# ]

# # === Step 4: MoverScore ===
# ref_idf = get_idf_dict(references)
# hyp_idf = get_idf_dict(candidates)

# df["MoverScore"] = [
#     word_mover_score([ref], [cand], ref_idf, hyp_idf, stop_words=[], n_gram=1, remove_subwords=True)[0]
#     for ref, cand in tqdm(zip(references, candidates), total=len(df), desc="MoverScore")
# ]

# === Step 5: GPTScore ===
# openai.api_key = os.getenv("OPENAI_API_KEY")  # Ensure this is set

# def gpt_score(reference, candidate):
#     prompt = f"""
# You are evaluating a candidate answer against a reference answer.

# Reference Answer: "{reference}"
# Candidate Answer: "{candidate}"

# Rate the candidate answer on a scale of 0 to 5, where 5 means the answer is perfectly aligned, and 0 means it's completely incorrect.

# Just return the score as a number.
# """
#     try:
#         client = openai.OpenAI()
#         response = client.chat.completions.create(
#             model="gpt-4",
#             messages=[
#                 {"role": "system", "content": "You are a fair and precise evaluator."},
#                 {"role": "user", "content": prompt}
#             ]
#         )
#         score_text = response["choices"][0]["message"]["content"].strip()
#         return float(score_text)
#     except Exception as e:
#         print(f"[ERROR]: {e}")
#         return None

# gpt_scores = []
# for ref, cand in tqdm(zip(references, candidates), total=len(df), desc="GPTScore"):
#     gpt_scores.append(gpt_score(ref, cand))
#     time.sleep(1)  # To avoid hitting rate limits

# df["GPTScore"] = gpt_scores

# === Save Output ===
df.to_csv("../data/gpt-4o-mini_eval_with_bert.csv", index=False)
print("✅ All scores computed and saved to o3_eval_with_all_scores.csv")


In [ ]:
from bert_score import score
import pandas as pd

# Load CSV
df = pd.read_csv("o3_eval_output.csv")

# Get expected vs model answers as lists
references = df["Expected Answer"].astype(str).tolist()
candidates = df["Model Answer"].astype(str).tolist()

# Compute BERTScore (returns three lists)
P, R, F1 = score(candidates, references, lang='en', verbose=True)

# Store F1 scores in a new column
df["BERT Score"] = F1

# (Optional) Save back to CSV
df.to_csv("o3_eval_with_bert.csv", index=False)


# ✨ Few-Shot - One-Shot Prompting
Here we use `FewShotPromptTemplate` from LangChain to provide **multiple Q&A examples**.


In [ ]:
import pandas as pd
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
import warnings
warnings.filterwarnings('ignore')

# === Load data ===
questions_df = pd.read_csv("../data/Benchmark Questions_GPT - Categorized.csv")
questions_df["Type"] = questions_df["Type"].astype(str).str.strip().str.lower()

examples_df = pd.read_csv("../data/cache_examples - list.csv")
examples_df["Type"] = examples_df["Type"].astype(str).str.strip().str.lower()

# === Few-shot helper ===
def get_examples_by_type(examples_df, question_type):
    qtype_clean = question_type.strip().lower()
    filtered = examples_df[examples_df["Type"] == qtype_clean]

    if filtered.empty:
        print(f"[WARN] No examples found for type '{qtype_clean}'. Available: {examples_df['Type'].unique()}")
    return [{"input": row["input"], "output": row["output"]} for _, row in filtered.iterrows()]

# === Main function ===
def ask_gpt_with_rag(question, question_type, processed_data, examples_df, mode="zero"):
    try:
        # === Always pull RAG context ===
        rag_context = process_query(question, processed_data, embedding_model)
        combined_context = f"{rag_context}\n\nNow answer this question.\nQuestion: {question}"

        if mode == "zero":
            prompt_text = combined_context + "\nThe correct answer is,"
            response = llm_chain.run({"prompt_text": prompt_text})

        elif mode in {"one", "few"}:
            examples = get_examples_by_type(examples_df, question_type or "default")
            if not examples:
                return combined_context, "[ERROR] No examples found for this type."

            example_prompt = PromptTemplate(
                input_variables=["input", "output"],
                template="Q: {input}\nA: {output}"
            )

            dynamic_prompt = FewShotPromptTemplate(
                examples=examples[:1] if mode == "one" else examples,
                example_prompt=example_prompt,
                suffix="Q: {context}\nA:",
                input_variables=["context"]
            )

            prompt_text = dynamic_prompt.format(context=combined_context)
            response = llm_chain.run({"prompt_text": prompt_text})

        else:
            return combined_context, "[ERROR] Unknown mode."

        return prompt_text, response

    except Exception as e:
        return question, f"[ERROR] {str(e)}"

# === Preview table ===
def preview_questions_table(questions_df, processed_data, examples_df, mode="zero", num_preview=5):
    rows = []
    for idx, row in questions_df.head(num_preview).iterrows():
        question = row["Question"]
        qtype = row["Type"]
        expected = row['Expected_Answer']
        print(f"Processing {idx+1}/{num_preview} [{mode}-shot | {qtype}]...")
        prompt, model_answer = ask_gpt_with_rag(question, qtype, processed_data, examples_df, mode=mode)
        rows.append({
            "Question": question,
            "Question Type": qtype,
            "Expected Answerd": expected,
            "Prompt Used": prompt,
            "Model Answer": model_answer
        })
    return pd.DataFrame(rows)

# === Run preview ===
pd.set_option("display.max_colwidth", None)
df_preview = preview_questions_table(questions_df, loaded_data, examples_df, mode="one", num_preview=1)
df_preview

In [ ]:
# === Run Full Scale ====
def run_and_save(questions_df, processed_data, examples_df, mode="few"):
    model_name = llm_chain.llm.model_name
    output_csv = f"../data/rag_eval_{mode}-shot_{model_name}.csv"

    rows = []
    for idx, row in questions_df.iterrows():
        question = row["Question"]
        qtype = row["Type"]
        expected = row['Expected_Answer']
        print(f"Processing {idx+1}/{len(questions_df)} [{mode}-shot | {qtype}]...")
        prompt, model_answer = ask_gpt_with_rag(question, qtype, processed_data, examples_df, mode=mode)
        rows.append({
            "Question": question,
            "Question Type": qtype,
            "Expected Answer": expected,
            "Prompt Used": prompt,
            "Model Answer": model_answer
        })

    df_full = pd.DataFrame(rows)
    df_full.to_csv(output_csv, index=False)
    print(f"✅ Done! Results saved to: {output_csv}")

# === Run it ===
run_and_save(questions_df, loaded_data, examples_df, mode="zero")


# Chatbot - Models with memory


In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOpenAI
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

from rag_source import *

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# OpenAI LLM
llm = ChatOpenAI(
    model_name="gpt-5",
    temperature=1,
    max_completion_tokens=5000,
)

# Memory buffer
memory = ConversationBufferMemory(memory_key="history", return_messages=True)

# Conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False  # Optional: shows messages being sent
)

In [ ]:
def chatbot_loop(processed_data):
    print("🔁 RAG Chatbot with Memory (type 'exit' to stop)")
    while True:
        query = input("\nUser: ")
        if query.strip().lower() in {"exit", "quit", "bye"}:
            print("👋 Exiting the chatbot. Goodbye!")
            break

        # Process query to build context-aware prompt
        prompt_text = process_query(query, processed_data, embedding_model)
        prompt_text += " The correct answer is:"
        
        # prompt_text = analyze_text_prompt_llm

        # Run the chatbot with memory
        response = conversation.run(prompt_text)

        print("\nUser:", query)
        print("\nAssistant:", response)

In [ ]:
chatbot_loop(loaded_data)

In [ ]:
for m in memory.chat_memory.messages:
    print(f"{m.type.upper()}: {m.content}")

In [ ]:
############ LLM RETRIEVER CHATBOT #########

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOpenAI
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

from rag_source import *

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# OpenAI LLM
llm = ChatOpenAI(
    model_name="gpt-5",
    temperature=1,
    max_completion_tokens=10000,
)

# Memory buffer
memory = ConversationBufferMemory(memory_key="history", return_messages=True)

# Conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False  # Optional: shows messages being sent
)

In [ ]:

def chatbot_loop(processed_data, model_name="gpt-5"):
    llm_chain = build_llm_chain_with_memory(model_name=model_name)

    print("🔁 Chatbot (LLM Retriever + Memory). Type 'exit' to stop.")
    while True:
        query = input("\nUser: ").strip()
        if query.lower() in {"exit", "quit", "bye"}:
            print("👋 Exiting the chatbot. Goodbye!")
            break

        # 1) Retrieval (your code-gen retriever)
        context_result, generated_code = llm_code_retrieval(query, processed_data)

        if not isinstance(context_result, str):
            try:
                context_result = str(context_result)
            except Exception as e:
                context_result = f"[Unable to convert context_result to string: {e}]"

        # Optional: debug
        # print("\n[DEBUG] Generated retriever code:\n", generated_code)

        # 2) Build final prompt for answer model
        prompt_text = prompt_gen(context_result, query)

        # 3) Answer with memory
        try:
            response = llm_chain.run({"prompt_text": prompt_text})
        except Exception as e:
            response = f"Generation failed: {e}"

        print("\nUser:", query)
        print("\nAssistant:", response)

In [ ]:
chatbot_loop(loaded_data)

In [ ]:
for m in memory.chat_memory.messages:
    print(f"{m.type.upper()}: {m.content}")

# Llama Index RAG

In [ ]:
import re
import pandas as pd
from typing import Dict, Any, List, Optional

from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI


def _safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)


def build_documents_from_loaded_data(
    loaded_data: Dict[str, Any],
    max_rows_per_trace: Optional[int] = 20000,
    row_text_fields: Optional[List[str]] = None,
) -> List[Document]:
    """
    Turns your dict-of-traces into LlamaIndex Documents.

    loaded_data[trace_id] = {
        "data_frame": pd.DataFrame,
        "metadata": str,
        "description": str
    }
    """
    docs: List[Document] = []

    if row_text_fields is None:
        row_text_fields = [
            "program_counter",
            "memory_address",
            "cache_set_id",
            "evict",
            "miss_type",
            "evicted_address",
            "accessed_address_recency_numeric",
            "accessed_address_reuse_distance_numeric",
            "evicted_address_reuse_distance_numeric",
            "function_name",
        ]

    for trace_id, trace_obj in loaded_data.items():
        df = trace_obj.get("data_frame", None)
        meta = _safe_str(trace_obj.get("metadata", ""))
        desc = _safe_str(trace_obj.get("description", ""))

        # 1) Trace-level summary document
        summary_text = f"TRACE_ID: {trace_id}\nDESCRIPTION: {desc}\nMETADATA: {meta}"
        docs.append(
            Document(
                text=summary_text,
                metadata={"trace_id": trace_id, "doc_type": "trace_summary"},
            )
        )

        if df is None or not isinstance(df, pd.DataFrame) or df.empty:
            continue

        # Optional downsample to keep index size manageable
        if max_rows_per_trace is not None and len(df) > max_rows_per_trace:
            df_use = df.sample(n=max_rows_per_trace, random_state=0)
        else:
            df_use = df

        # 2) Row-level documents
        for idx, row in df_use.iterrows():
            # Build a compact, searchable row string
            parts = []
            for col in row_text_fields:
                if col in df_use.columns:
                    parts.append(f"{col}={_safe_str(row.get(col))}")

            row_text = f"TRACE_ID: {trace_id}\nROW_INDEX: {idx}\n" + ", ".join(parts)

            # Add a little extra if available (but avoid huge blobs)
            if "assembly_code" in df_use.columns:
                asm = _safe_str(row.get("assembly_code"))
                if asm:
                    asm = asm[:800]  # cap
                    row_text += f"\nassembly_code_snip: {asm}"

            if "function_code" in df_use.columns:
                fc = _safe_str(row.get("function_code"))
                if fc:
                    fc = fc[:800]
                    row_text += f"\nfunction_code_snip: {fc}"

            docs.append(
                Document(
                    text=row_text,
                    metadata={
                        "trace_id": trace_id,
                        "doc_type": "trace_row",
                        "row_index": int(idx) if str(idx).isdigit() else _safe_str(idx),
                    },
                )
            )

    return docs


def build_llamaindex_retriever_from_loaded_data(
    loaded_data: Dict[str, Any],
    top_k: int = 8,
    max_rows_per_trace: Optional[int] = 20000,
):
    """
    Returns a LlamaIndex retriever you can call with:
    nodes = retriever.retrieve("your question")
    """
    # Configure LlamaIndex global settings (OpenAI example)
    Settings.llm = OpenAI(model="gpt-5", temperature=0.8)
    Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

    docs = build_documents_from_loaded_data(
        loaded_data,
        max_rows_per_trace=max_rows_per_trace,
    )

    splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
    index = VectorStoreIndex.from_documents(docs, transformations=[splitter])

    retriever = index.as_retriever(similarity_top_k=top_k)
    return retriever, index

In [ ]:
# loaded_data is your dict database (old format)
retriever, index = build_llamaindex_retriever_from_loaded_data(
    loaded_data,
    top_k=10,
    max_rows_per_trace=5000,
)

# nodes = retriever.retrieve("For PC 0x401e31 in lbm lru, are accesses mostly hits or misses?")
# for n in nodes[:5]:
#     print(n.score, n.metadata)
#     print(n.text[:300], "\n---\n")

In [ ]:
nodes = retriever.retrieve("For astar + Parrot, which memory address is evicted most frequently?")
for n in nodes[:5]:
    print(n.score, n.metadata)
    print(n.text[:5000], "\n---\n")

In [ ]:
####### Example usage #######
query = "For astar + Parrot, which memory address is evicted most frequently?"
result, code = llm_code_retrieval(query, processed_data=loaded_data)

print("Generated Code:\n", code)
print("\nRetrieved Result:\n", result)